In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl
import numpy as np
import pickle
import gc
from deepctr.feature_column import SparseFeat, VarLenSparseFeat
import faiss

# from preprocess import (
#     gen_data_set,
#     gen_model_input,
#     gen_data_set_sdm,
#     gen_model_input_sdm,
# )
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model

from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss, NegativeSampler
import sys

sys.path.append("../src/otto_recommender")
import utils
import deepmatch_preprocess
import similar_embs

In [2]:
import tensorflow as tf

print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

Num GPUs Available:  1


In [3]:
import tensorflow

tensorflow.__version__

'2.8.0'

In [4]:
from tqdm.notebook import tqdm

tqdm.pandas()

In [5]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 5
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch13_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    new_split = ""
    count_aid_trs = 60


config = Config()

In [6]:
dict_types = dict()
dict_types["clicks"] = 0
dict_types["carts"] = 1
dict_types["orders"] = 2

type_labels = pl.DataFrame(
    {"type": list(dict_types.keys()), "values": list(dict_types.values())}
)

In [13]:
from deepctr.layers import SequencePoolingLayer, Concat, NoMask, DNN
from deepmatch.layers.core import InBatchSoftmaxLayer

In [18]:
%%time

with tf.device("CPU"):
    model = tf.keras.models.load_model(
        'dssm_epoch_try',
        custom_objects={
            'SequencePoolingLayer': SequencePoolingLayer, 'Concat': Concat, 'NoMask': NoMask, 'DNN': DNN, 'InBatchSoftmaxLayer': InBatchSoftmaxLayer
        }
    )

IndexError: list index out of range

In [7]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, new_split=False, count_aid_trs=config.count_aid_trs)
pad = ordinal_encoder.unknown_value

X_val_seq = (
    X_val_seq.join(type_labels, on=config.col_type, how="left")
    .drop(columns=[config.col_type])
    .rename({"values": config.col_type})
)

X_val_seq = X_val_seq.to_pandas()
val_set = deepmatch_preprocess.gen_data_set(X_val_seq, seq_max_len=config.max_len, pad=pad, mode='val')
with open(f'val_set_dssm_{config.count_aid_trs}.pickle', 'wb') as f:
    pickle.dump(val_set, f, protocol=4)
X_val_seq = pl.from_pandas(X_val_seq)

X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
    config,
    X_train,
    X_val,
    X_val_seq,
    ordinal_encoder,
    return_only_val=True,
    truncate_val=True,
    return_just_splitted=True,
)

# X_train = X_train.drop("count")
# X_train = X_train.drop("n_session_right")
# X_val = X_val.drop("count")
# X_val_seq = X_val_seq.drop("count")
X_val_seq = X_val_seq.sort("user")
val_seq_data = X_val_seq.to_pandas()[config.col_aid].tolist()

del X_train
gc.collect()

100%|████████████████████████████████| 466825/466825 [00:44<00:00, 10424.85it/s]


7


  0%|          | 0/466825 [00:00<?, ?it/s]

CPU times: user 15min 3s, sys: 1min 55s, total: 16min 58s
Wall time: 3min 55s


18

In [8]:
%%time

with open(f'train_set_dssm_{config.count_aid_trs}.pickle', 'rb') as f:
    train_set = pickle.load(f)

CPU times: user 1min 40s, sys: 16.9 s, total: 1min 57s
Wall time: 1min 57s


In [9]:
len(train_set), len(train_set[0]), len(train_set[-1])

(45375280, 7, 7)

In [10]:
SEQ_LEN = config.max_len
SEQ_LEN_short = 3
SEQ_LEN_prefer = 10
negsample = 0

In [11]:
%%time

# with open(f'lbe_user_{config.count_aid_trs}.pickle', 'rb') as f:
#     lbe_user = pickle.load(f)

with open(f'lbe_item_{config.count_aid_trs}.pickle', 'rb') as f:
    lbe_item = pickle.load(f)

CPU times: user 1.2 ms, sys: 161 µs, total: 1.36 ms
Wall time: 1.17 ms


In [12]:
item_pad = lbe_item.unknown_value
item_pad

453724

In [13]:
%%time

train_model_input, train_label = deepmatch_preprocess.gen_model_input(
    train_set, 
    seq_max_len=SEQ_LEN, 
    item_pad=item_pad, 
    type_pad=3
)

extracting arrays
padding
CPU times: user 4min 36s, sys: 22.6 s, total: 4min 58s
Wall time: 4min 59s


In [14]:
del train_set
gc.collect()

162

In [15]:
%%time

with open(f'val_set_dssm_{config.count_aid_trs}.pickle', 'rb') as f:
    val_set = pickle.load(f)

CPU times: user 1.2 s, sys: 107 ms, total: 1.31 s
Wall time: 1.31 s


In [16]:
%%time

val_model_input, _ = deepmatch_preprocess.gen_model_input(
    val_set, 
    seq_max_len=SEQ_LEN, 
    item_pad=item_pad, 
    type_pad=3
)

extracting arrays
padding
CPU times: user 2.35 s, sys: 45.5 ms, total: 2.4 s
Wall time: 2.33 s


In [17]:
del val_set
gc.collect()

380

In [18]:
val_seq_data[:2]

[array([279097, 142193]),
 array([34876, 48616, 14199, 47407, 48616, 48616, 48616, 48616])]

In [19]:
val_model_input

{'item_id': array([142193,  48616, 116970, ..., 257328, 258853,  66352]),
 'hist_item_id': array([[279097, 453724, 453724, 453724, 453724],
        [ 48616,  48616,  48616,  47407,  14199],
        [203705, 252449, 145485, 246175, 203705],
        ...,
        [453724, 453724, 453724, 453724, 453724],
        [453724, 453724, 453724, 453724, 453724],
        [453724, 453724, 453724, 453724, 453724]], dtype=int32),
 'hist_types': array([[0, 3, 3, 3, 3],
        [1, 0, 0, 0, 0],
        [2, 2, 2, 2, 1],
        ...,
        [3, 3, 3, 3, 3],
        [3, 3, 3, 3, 3],
        [3, 3, 3, 3, 3]], dtype=int32),
 'hist_len': array([1, 5, 5, ..., 0, 0, 0])}

In [20]:
%%time

embedding_dim = 32

user_feature_columns = [
    # SparseFeat('user_id', 13366604 + 1, 16),
    VarLenSparseFeat(SparseFeat('hist_item_id', item_pad + 1, embedding_dim, embedding_name="item_id"), SEQ_LEN, 'mean', 'hist_len'),
    VarLenSparseFeat(SparseFeat('hist_types', 3 + 1, embedding_dim, embedding_name="types"), SEQ_LEN, 'mean', 'hist_len'),
    ]


item_feature_columns = [SparseFeat('item_id', item_pad + 1, embedding_dim),
                        #SparseFeat('types', 2 + 1, embedding_dim)
                       ]

CPU times: user 110 µs, sys: 14 µs, total: 124 µs
Wall time: 133 µs


In [21]:
%%time

from collections import Counter
train_counter = Counter(train_model_input['item_id'])
item_count = [train_counter.get(i, 0) for i in range(item_feature_columns[0].vocabulary_size)]
sampler_config = NegativeSampler('inbatch', num_sampled=64, item_name="item_id", item_count=item_count)

CPU times: user 11.6 s, sys: 59.7 ms, total: 11.7 s
Wall time: 11.6 s


In [22]:
# lens = [
#     len(train_model_input["hist_types"][i])
#     for i in tqdm(range(len(train_model_input["hist_types"])))
# ]

In [23]:
from keras.optimizers import adam_v2

optimizer = adam_v2.Adam(learning_rate=0.001)

In [24]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

Num GPUs Available:  1


In [25]:
from tensorflow.python.client import device_lib

print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 15707097842126187551
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 10420486144
locality {
  bus_id: 3
  numa_node: 2
  links {
  }
}
incarnation: 18122719424112927481
physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1080 Ti, pci bus id: 0000:41:00.0, compute capability: 6.1"
xla_global_id: 416903419
]


2023-01-25 21:29:23.428473: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-01-25 21:29:23.836635: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /device:GPU:0 with 9937 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080 Ti, pci bus id: 0000:41:00.0, compute capability: 6.1


In [26]:
# # Prepare the training dataset.
# train_dataset = tf.data.Dataset.from_tensor_slices((train_model_input, train_label))
# train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size)

# # # Prepare the validation dataset.
# # val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))
# # val_dataset = val_dataset.batch(batch_size)

In [27]:
# %%time

# # 3.Define Model and train

# import tensorflow as tf
# # if tf.__version__ >= '2.0.0':
# #     tf.compat.v1.disable_eager_execution()
# # else:
# #     K.set_learning_phase(True)

# tf.random.set_seed(1337)

# model = DSSM(user_feature_columns, item_feature_columns, user_dnn_hidden_units=(128, 64, embedding_dim),
#              item_dnn_hidden_units=(64, embedding_dim,), loss_type='softmax', sampler_config=sampler_config)

# loss_fn = sampledsoftmaxloss

In [28]:
batch_size = 512

tf.random.set_seed(1337)

In [29]:
%%time

SHUFFLE_BUFFER_SIZE = 4096
# with tf.device("CPU"):
#     train_dataset = tf.data.Dataset.from_tensor_slices((train_model_input, train_label))
#     train_dataset = train_dataset.shuffle(SHUFFLE_BUFFER_SIZE).batch(batch_size)

train_dataset = tf.data.Dataset.from_tensor_slices((train_model_input, train_label))
train_dataset = train_dataset.shuffle(SHUFFLE_BUFFER_SIZE, seed=1337).batch(batch_size)

# test_dataset = tf.data.Dataset.from_tensor_slices((X_valid, y_valid))
# validation_dataset = test_dataset.batch(BATCH_SIZE)

2023-01-25 21:29:23.961948: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9937 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1080 Ti, pci bus id: 0000:41:00.0, compute capability: 6.1


CPU times: user 464 ms, sys: 1.11 s, total: 1.58 s
Wall time: 1.56 s


In [30]:
# %%time

# for step, (X_batch, y_batch) in tqdm(enumerate(train_dataset), total=len(train_dataset)):
#     continue
## 3min 15s

In [31]:
# tf.debugging.set_log_device_placement(True)

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

optimizer = Adam(lr=0.001)
criterion = sampledsoftmaxloss
metric = sampledsoftmaxloss


def apply_gradients(X, y, model, loss_obj, optimizer):
    """
    Args:
    X: The training data
    y:  training labels
    model: The model being trained
    loss_obj: loss object
    """
    tf.debugging.set_log_device_placement(True)
    with tf.device(tf.DeviceSpec(device_type="GPU", device_index=0)):
        with tf.GradientTape() as t:
            # calculate predictions using the model
            predictions = model(X)
            # calculate the loss using the loss object
            loss = loss_obj(y, predictions)
    # computing the gradients of the loss with respect to the model weights
    gradients = t.gradient(loss, model.trainable_weights)
    # Apply the gradients to update the model weights
    optimizer.apply_gradients(zip(gradients, model.trainable_weights))

    return predictions, loss


# def validate(validation_dataset, model, loss_obj, metric):
#     """
#     Args:
#     validation_dataset : validation  dataset consisting X, y
#     model : model being validated
#     loss_obj: Loss function object
#     metric: metric used to measure the model's performance
#     """
#     # Create an arrya to store the losses
#     losses = []
#     # Loop over X,y in the validation dataset
#     for (X, y) in validation_dataset:
#         # Calculate the predictions
#         predictions = model.predict(X)
#         # calculate the loss
#         loss = loss_obj(y, predictions)
#         # append the loss into the losses list
#         losses.append(loss)
#         # Calculate the validaiton accuracy
#         validation_accuracy = metric(y, predictions)
#     return losses


@utils.timeit
def create_index(model, lbe_item, k, val_model_input=val_model_input):
    items = (
        lbe_item.transform(lbe_item.categories_[0].reshape(-1, 1))
        .reshape(1, -1)[0]
        .astype(np.int32)
    )
    item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)
    all_item_model_input = {"item_id": items}
    item_embs = item_embedding_model.predict(
        all_item_model_input, batch_size=2**12, verbose=1
    )
    user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
    user_embs = user_embedding_model.predict(
        val_model_input, batch_size=2**12, verbose=1
    )

    index = faiss.IndexFlatL2(32)
    # make it a flat GPU index
    # gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, index)
    index.add(item_embs)
    # we want to see k nearest neighbors
    S, I = index.search(user_embs, k)
    S = 1 - S

    dict_sims = dict()
    for i in tqdm(range(len(S))):
        dict_sims[i] = (S[i], I[i])
    return dict_sims


@utils.timeit
def func_valid(
    model,
    lbe_item=lbe_item,
    val_seq_data=val_seq_data,
    X_val_seq=X_val_seq,
    X_val=X_val,
    mod="dssm_sim_items",
    num_recs=20,
):
    print("creating index")
    dict_sims = create_index(model, lbe_item=lbe_item, k=num_recs)

    rec_items = [dict_sims[key][1] for key in dict_sims]
    rec_scores = [dict_sims[key][0] for key in dict_sims]
    recs = pl.DataFrame(
        {
            "index": utils.get_sequential_index(rec_items),
            "item_id": [item for rec in rec_items for item in rec],
            "similar_score": [score for rec in rec_scores for score in rec],
            "similar_rank": [
                rnk + 1 for rec in rec_scores for rnk, _ in enumerate(rec)
            ],
        }
    )

    df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
    df_idx["user"] = X_val_seq.to_pandas()["user"].astype(int)
    df_idx = pl.from_pandas(df_idx)
    recs = recs.join(df_idx, on="index")

    recs = recs.to_pandas()
    recs["item_id"] = lbe_item.inverse_transform(
        recs["item_id"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    recs = recs.dropna()
    recs = pl.from_pandas(recs)

    recs = recs.rename({"similar_score": f"{mod}_score", "similar_rank": f"{mod}_rank"})

    # recs.write_parquet(
    #     config.cur_model_output_path
    #     + f"cur_{mod}_df_val_preds_{config.num_recs}{config.new_split}.parquet"
    # )

    recs = (
        recs.groupby(["index"])
        .agg(
            [
                pl.col("item_id").list(),
            ]
        )
        .sort("index")
        .to_pandas()
    )

    recs["user"] = X_val_seq.to_pandas()["user"]
    recs["item_id"] = recs["item_id"].progress_apply(
        lambda x: utils.truncate_list(list(x), from_start=True, max_len=20)
    )
    recs = recs.merge(X_val[["user", "target"]].to_pandas())

    preds = recs["item_id"].tolist()
    preds = [list(pred) for pred in preds]

    targets = recs["target"].tolist()
    targets = [list(target) for target in targets]

    score = utils.recall_several_targets(preds, targets)
    print("score:", score)
    return score, dict_sims


# @utils.timeit
# def func_valid(
#     model,
#     lbe_item=lbe_item,
#     val_seq_data=val_seq_data,
#     X_val_seq=X_val_seq,
#     X_val=X_val,
#     mod="dssm_sim_items",
#     num_recs=20,
# ):
#     print("creating index")
#     dict_sims = create_index(model, lbe_item=lbe_item, k=num_recs)

#     recommender = similar_embs.SimilarEmbeddingsRecommender(dict_sims=dict_sims)

#     recs = recommender.recommend(val_seq_data, n_recs=num_recs, last_k_items=10)

#     df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
#     df_idx["user"] = X_val_seq.to_pandas()["user"].astype(int)
#     df_idx = pl.from_pandas(df_idx)
#     recs = recs.join(df_idx, on="index")

#     recs = recs.to_pandas()
#     recs["item_id"] = lbe_item.inverse_transform(
#         recs["item_id"].values.reshape(-1, 1)
#     ).reshape(1, -1)[0]
#     recs = recs.dropna()
#     recs = pl.from_pandas(recs)

#     recs = recs.rename({"similar_score": f"{mod}_score", "similar_rank": f"{mod}_rank"})

#     # recs.write_parquet(
#     #     config.cur_model_output_path
#     #     + f"cur_{mod}_df_val_preds_{config.num_recs}{config.new_split}.parquet"
#     # )

#     recs = (
#         recs.groupby(["index"])
#         .agg(
#             [
#                 pl.col("item_id").list(),
#             ]
#         )
#         .sort("index")
#         .to_pandas()
#     )

#     recs["user"] = X_val_seq.to_pandas()["user"]
#     recs["item_id"] = recs["item_id"].progress_apply(
#         lambda x: utils.truncate_list(list(x), from_start=True, max_len=20)
#     )
#     recs = recs.merge(X_val[["user", "target"]].to_pandas())

#     preds = recs["item_id"].tolist()
#     preds = [list(pred) for pred in preds]

#     targets = recs["target"].tolist()
#     targets = [list(target) for target in targets]

#     score = utils.recall_several_targets(preds, targets)
#     print("score:", score)
#     return score, dict_sims


def train_one_step(
    train_dataset, optimizer, model, metric_obj, loss_obj, num_batches_to_validate=None
):
    """
    Args:
    training_dataset: tensorflow dataset consisting of (X,y)
    optimizer: optimizer object used to optimize the network
    model: The model being trained
    metric: metric used to measure the model's performance
    """
    # Create an empty list to store the losses
    losses = []
    # Create a progress bar to keep track of the training
    pbar = tqdm(
        train_dataset,
        total=len(train_dataset),
    )
    # loss over each batch of X,y in the training dataset
    for batch_idx, (X_batch, y_batch) in enumerate(pbar):
        # Calculate predictions and loss
        predictions, loss = apply_gradients(
            X_batch, y_batch, model, loss_obj, optimizer
        )
        # append the loss to the losses list
        losses.append(loss)
        pbar.set_description("Training loss for step %s: %.4f" % (batch_idx, loss))
        # pbar.update()
        # Calculate the metric
        # metric_obj(y_batch, predictions)
        # print(model.item_embedding.shape)
        if num_batches_to_validate:
            if batch_idx % num_batches_to_validate == 0 and batch_idx > 0:
                print("batch_idx:", batch_idx)
                score, dict_sims = func_valid(model)
                print("_" * 10)
                print()
    score, dict_sims = func_valid(model)
    loss_train = np.mean(losses)
    return loss_train


def train(
    train_dataset,
    validaiton_dataset,
    epochs,
    model,
    optimizer,
    loss_obj,
    train_metric_obj,
    validation_metric_obj,
):
    """
    Args:
    train_dataset: training_dataset consisting of X,y
    validation_dataset: validation_dataset consisting of X,y
    epochs: number of epochs to train the model for
    model: model being trained
    optimizer: optimizer used for optimizing the model
    loss: loss object used to measure the loss
    train_metric_obj: training metric used to evaluate the model on the training dataset
    train_metric_obj: training metric used to evaluate the model on the validation dataset
    """
    # Loop for range of epochs
    for epoch in range(epochs):
        # Compute training loss
        train_losses = train_one_step(
            train_dataset, optimizer, model, train_metric_obj, loss_obj
        )
        # Compute validation loss
        validation_losses = validate(
            validation_dataset, model, loss_obj, validation_metric_obj
        )
        # Compute the mean of the validaiton losses
        validation_losses_mean = np.mean(validation_losses)
        # Compute the mean of the training losses
        train_losses_mean = np.mean(train_losses)
        # Compute training accuracy
        train_accuracy = train_metric_obj.result()
        # Compute validation accuracy
        validation_accuracy = validation_metric_obj.result()
        print(
            "\n Epoch %s: Train loss: %.4f  Validation Loss: %.4f, Train Accuracy: %.4f, Validation Accuracy %.4f"
            % (
                epoch + 1,
                float(train_losses_mean),
                float(validation_losses_mean),
                float(train_accuracy),
                float(validation_accuracy),
            )
        )
        # Reset the metrics objects
        train_metric_obj.reset_states()
        validation_metric_obj.reset_states()

In [32]:
%%time
with tf.device('/device:GPU:0'):
    model = DSSM(user_feature_columns, item_feature_columns, user_dnn_hidden_units=(64, embedding_dim),
             item_dnn_hidden_units=(64, embedding_dim), loss_type='softmax', sampler_config=sampler_config, dnn_dropout=0.1)

CPU times: user 3.94 s, sys: 42.8 ms, total: 3.98 s
Wall time: 4.01 s


In [33]:
%%time

tf.debugging.set_log_device_placement(True)

with tf.device('/device:GPU:0'):
    train_one_step(train_dataset, optimizer, model, metric_obj=criterion, loss_obj=criterion, num_batches_to_validate=10_000)

  0%|          | 0/88624 [00:00<?, ?it/s]

batch_idx: 10000
creating index
114/114 [==============================] - 0s 1ms/step


  0%|          | 0/466825 [00:00<?, ?it/s]

func 'create_index' took: 20.1561 min


  0%|          | 0/466825 [00:00<?, ?it/s]

score: 0.1342730145129331
func 'func_valid' took: 20.4639 min
__________

batch_idx: 20000
creating index
114/114 [==============================] - 0s 1ms/step


... (28 outputs left out)

  0%|          | 0/466825 [00:00<?, ?it/s]

func 'create_index' took: 20.1192 min


  0%|          | 0/466825 [00:00<?, ?it/s]

score: 0.17817454792124102
func 'func_valid' took: 20.3984 min
CPU times: user 1d 9h 47min 39s, sys: 15min 7s, total: 1d 10h 2min 46s
Wall time: 21h 29min 20s


In [38]:
%%time

model.save(f"dssm_epoch_{1}.h5")

CPU times: user 3.1 s, sys: 79.9 ms, total: 3.18 s
Wall time: 3.18 s


In [34]:
stop

NameError: name 'stop' is not defined

In [ ]:
%%time

items = lbe_item.transform(lbe_item.categories_[0].reshape(-1, 1)).reshape(1, -1)[0].astype(np.int32)
item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)
all_item_model_input = {"item_id": items}
item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12, verbose=1)

In [ ]:
item_embs.shape

In [ ]:
user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
user_embs = user_embedding_model.predict(val_model_input, batch_size=2**12, verbose=1)

In [ ]:
val_model_input

In [ ]:
%%time

import faiss  # make faiss available

# res = faiss.StandardGpuResources()  # use a single GPU
index = faiss.IndexFlatL2(32)
# make it a flat GPU index
# gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, index)
index.add(item_embs)

In [ ]:
# 500 - 1.55 sec
# 1000 - 2.51 sec
# 2000 - 5.08 sec

In [ ]:
2.51 / 1.55

In [ ]:
5.08 / 2.51

In [ ]:
%%time

k = 100 # we want to see 100 nearest neighbors
S_t, I_t = index.search(user_embs[:2000], k)
S_t = 1 - S_t

In [ ]:
%%time

# items_real = lbe_item.categories_[0]
# assert len(items_real) == len(S)
dict_sims = dict()
for i in tqdm(range(len(S_t))):
    dict_sims[i] = (S_t[i], I_t[i])

In [ ]:
recs = [dict_sims[key][0] for key in dict_sims]

In [ ]:
rec_items = [dict_sims[key][1] for key in dict_sims]
rec_scores = [dict_sims[key][0] for key in dict_sims]
recs = pl.DataFrame(
    {
        "index": utils.get_sequential_index(rec_items),
        "item_id": [item for rec in rec_items for item in rec],
        "similar_score": [score for rec in rec_scores for score in rec],
        "similar_rank": [rnk + 1 for rec in rec_scores for rnk, _ in enumerate(rec)],
    }
)

In [ ]:
%%time

k = 20                          # we want to see 20 nearest neighbors
S, I = index.search(item_embs, k)
S = 1 - S

In [ ]:
%%time

items_real = lbe_item.categories_[0]
assert len(items_real) == len(S)
dict_sims = dict()
for i in tqdm(range(len(S))):
    dict_sims[i] = (S[i], I[i])

In [ ]:
def func_valid(
    model,
    dict_sims,
    lbe_item=lbe_item,
    val_seq_data=val_seq_data,
    X_val_seq=X_val_seq,
    X_val=X_val,
    mod="dssm_sim_items",
    num_recs=20,
):
    items = (
        lbe_item.transform(lbe_item.categories_[0].reshape(-1, 1))
        .reshape(1, -1)[0]
        .astype(np.int32)
    )
    item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)
    all_item_model_input = {"item_id": items}
    item_embs = item_embedding_model.predict(
        all_item_model_input, batch_size=2**12, verbose=1
    )

    recommender = similar_embs.SimilarEmbeddingsRecommender(dict_sims=dict_sims)

    recs = recommender.recommend(val_seq_data, n_recs=num_recs, last_k_items=10)

    df_idx = recs.to_pandas()[["index"]].drop_duplicates().reset_index(drop=True)
    df_idx["user"] = X_val_seq.to_pandas()["user"].astype(int)
    df_idx = pl.from_pandas(df_idx)
    recs = recs.join(df_idx, on="index")

    recs = recs.to_pandas()
    recs["item_id"] = ordinal_encoder.inverse_transform(
        recs["item_id"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    recs = recs.dropna()
    recs = pl.from_pandas(recs)

    recs = recs.rename({"similar_score": f"{mod}_score", "similar_rank": f"{mod}_rank"})

    # recs.write_parquet(
    #     config.cur_model_output_path
    #     + f"cur_{mod}_df_val_preds_{config.num_recs}{config.new_split}.parquet"
    # )

    recs = (
        recs.groupby(["index"])
        .agg(
            [
                pl.col("item_id").list(),
            ]
        )
        .sort("index")
        .to_pandas()
    )

    recs["user"] = X_val_seq.to_pandas()["user"]
    recs["item_id"] = recs["item_id"].progress_apply(
        lambda x: utils.truncate_list(list(x), from_start=True, max_len=20)
    )
    recs = recs.merge(X_val[["user", "target"]].to_pandas())

    preds = recs["item_id"].tolist()
    preds = [list(pred) for pred in preds]

    targets = recs["target"].tolist()
    targets = [list(target) for target in targets]

    score = utils.recall_several_targets(preds, targets)
    print("score:", score)
    return score

In [ ]:
%%time

func_valid(model, dict_sims)

In [ ]:
dict_sims[517642]

In [ ]:
I

In [ ]:
S

In [ ]:
I

In [ ]:
stop

In [ ]:
stop

In [ ]:
%%time

train_one_step(train_dataset, optimizer, model, metric_obj=criterion, loss_obj=metric)

In [ ]:
stop

In [ ]:
%%time

# 3.Define Model and train

import tensorflow as tf
# if tf.__version__ >= '2.0.0':
#     tf.compat.v1.disable_eager_execution()
# else:
#     K.set_learning_phase(True)

tf.random.set_seed(1337)

model = DSSM(user_feature_columns, item_feature_columns, user_dnn_hidden_units=(128, 64, embedding_dim),
             item_dnn_hidden_units=(64, embedding_dim,), loss_type='softmax', sampler_config=sampler_config)

model.compile(optimizer='adam', loss=sampledsoftmaxloss)

history = model.fit(train_model_input, train_label,  # train_label,
                    batch_size=batch_size, epochs=1, verbose=1, validation_split=0.00, workers=10, use_multiprocessing=True)

In [ ]:
stop

In [ ]:
train_set[-2]

In [ ]:
dict_types = dict()
dict_types["clicks"] = 0
dict_types["carts"] = 1
dict_types["orders"] = 2

type_labels = pl.DataFrame(
    {"type": list(dict_types.keys()), "values": list(dict_types.values())}
)

In [ ]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, new_split=False)
# X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
#     config, 
#     X_train, 
#     X_val, 
#     X_val_seq, 
#     ordinal_encoder, 
#     return_only_val=True, 
#     truncate_val=True
# )
pad = ordinal_encoder.unknown_value

# X_train = X_train.drop("count")
# X_train = X_train.drop("n_session_right")
# X_val = X_val.drop("count")
# X_val_seq = X_val_seq.drop("count")

In [ ]:
%%time

X_train = (
    X_train.join(type_labels, on=config.col_type, how="left")
    .drop(columns=[config.col_type])
    .rename({"values": config.col_type})
)

In [ ]:
# %%time

# X_train = (
#     X_train.groupby([config.col_user])
#     .agg(
#         [
#             pl.col(config.col_aid).list(),
#             pl.col(config.col_type).list(),

#         ]
#     ).filter(pl.col(config.col_aid).apply(lambda x: len(x)) >= 2)
# )

In [ ]:
# %%time

# X_train = X_train.to_pandas()
# X_train[config.col_train_seq] = X_train[config.col_aid].progress_apply(lambda x: x[:-1])
# X_train[config.col_target] = X_train[config.col_aid].progress_apply(lambda x: x[-1])

# X_train['train_seq_type'] = X_train[config.col_type].progress_apply(lambda x: x[:-1])
# X_train['target_type'] = X_train[config.col_type].progress_apply(lambda x: x[-1])
# X_train = pl.from_pandas(X_train)

In [ ]:
X_train["aid"].max()

In [ ]:
X_train["user"].max()

In [ ]:
%%time

X_train = X_train.with_columns(
        [
            pl.col(config.col_aid)
            .cast(pl.Int32, strict=False)
            .alias(config.col_aid),
            pl.col(config.col_user)
            .cast(pl.Int32, strict=False)
            .alias(config.col_user),
            pl.col(config.col_type)
            .cast(pl.Int8, strict=False)
            .alias(config.col_type),
        ]
    )

In [ ]:
X_train

In [ ]:
len(X_train[config.col_user].unique())

In [ ]:
X_train[config.col_user].max()

In [ ]:
X_train = X_train.to_pandas()

In [ ]:
%%time

SEQ_LEN = 10
SEQ_LEN_short = 3
SEQ_LEN_prefer = 10
negsample = 0

# 1.Label Encoding for sparse features, and process sequence features with `gen_date_set` and `gen_model_input`
dict_lbes = dict()

feature_max_idx = dict()
feature_max_idx[config.col_type] = 2
feature_max_idx[config.col_aid] = pad
for feature in [config.col_user]:
    unknown_value = len(X_train[feature].unique())
    lbe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=unknown_value)
    X_train[feature] = lbe.fit_transform(X_train[feature].values.reshape(-1, 1)).astype('int32')
    feature_max_idx[feature] = unknown_value
    dict_lbes[feature] = lbe

In [ ]:
X_train.head()

In [ ]:
X_train["type"]

In [ ]:
col_ts = config.col_ts
col_item = config.col_aid
col_type = config.col_type
col_user = config.col_user

In [ ]:
%%time

train_set = deepmatch_preprocess.gen_data_set_sdm(X_train, seq_short_max_len=2, seq_prefer_max_len=10)

In [ ]:
del X_train, X_val, X_val_seq
gc.collect()

In [ ]:
%%time

import pickle

with open('train_set.pickle', 'wb') as f:
    pickle.dump(train_set, f, protocol=4)

In [ ]:
stop

In [ ]:
data[data["user"] == 0].head(15)

In [ ]:
    # train_uid = np.array([line[0] for line in train_set])
    # train_iid = np.array([line[1] for line in train_set])
    # train_label = np.array([line[2] for line in train_set])
    # short_train_seq = [line[3] for line in train_set]
    # prefer_train_seq = [line[4] for line in train_set]
    # train_short_len = np.array([line[5] for line in train_set])
    # train_prefer_len = np.array([line[6] for line in train_set])
    # short_train_seq_genres = np.array([line[7] for line in train_set])
    # prefer_train_seq_genres = np.array([line[8] for line in train_set])

In [ ]:
train_set[13]

In [ ]:
item_id_types_map[954978]

In [ ]:
%%time

sparse_features = [
    "user",
    "aid",
    "type",
]

SEQ_LEN = 10
negsample = 0

feature_max_idx = dict()
feature_lbe = dict()
for feature in sparse_features:
    lbe = LabelEncoder()
    data[feature] = lbe.fit_transform(data[feature]) + 1
    feature_max_idx[feature] = data[feature].max() + 1
    feature_lbe[feature] = lbe

In [ ]:
feature_max_idx

In [ ]:
%%time

user_profile = data[["user", "aid", "type"]].drop_duplicates("user")

item_profile = data[["movie_id", "genres"]].drop_duplicates("movie_id")

user_profile.set_index("user_id", inplace=True)

user_item_list = data.groupby("user_id")["movie_id"].apply(list)

train_set, test_set = gen_data_set(data, SEQ_LEN, negsample)

train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN)
test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN)

In [ ]:
%%time

embedding_dim = 32

user_feature_columns = [SparseFeat('user_id', feature_max_idx['user_id'], 16),
                        SparseFeat("gender", feature_max_idx['gender'], 16),
                        SparseFeat("age", feature_max_idx['age'], 16),
                        SparseFeat("occupation", feature_max_idx['occupation'], 16),
                        SparseFeat("zip", feature_max_idx['zip'], 16),
                        VarLenSparseFeat(SparseFeat('hist_movie_id', feature_max_idx['movie_id'], embedding_dim,
                                                    embedding_name="movie_id"), SEQ_LEN, 'mean', 'hist_len'),
                        VarLenSparseFeat(SparseFeat('hist_genres', feature_max_idx['genres'], embedding_dim,
                                embedding_name="genres"), SEQ_LEN, 'mean', 'hist_len'),
                        ]

item_feature_columns = [SparseFeat('movie_id', feature_max_idx['movie_id'], embedding_dim),
                        SparseFeat('genres', feature_max_idx['genres'], embedding_dim)
                       ]

from collections import Counter
train_counter = Counter(train_model_input['movie_id'])
item_count = [train_counter.get(i,0) for i in range(item_feature_columns[0].vocabulary_size)]
sampler_config = NegativeSampler('inbatch', num_sampled=255, item_name="movie_id", item_count=item_count)

In [ ]:
%%time

import tensorflow as tf
if tf.__version__ >= '2.0.0':
    tf.compat.v1.disable_eager_execution()
else:
    K.set_learning_phase(True)
    
model = DSSM(user_feature_columns, item_feature_columns,user_dnn_hidden_units=(128,64, embedding_dim),
             item_dnn_hidden_units=(64, embedding_dim,),loss_type='softmax',sampler_config=sampler_config)

model.compile(optimizer="adam", loss=sampledsoftmaxloss)

history = model.fit(train_model_input, train_label,  # train_label,
                    batch_size=256, epochs=3, verbose=1, validation_split=0.0, )

# 4. Generate user features for testing and full item features for retrieval
test_user_model_input = test_model_input
all_item_model_input = {"movie_id": item_profile['movie_id'].values,"genres": item_profile['genres'].values}

user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)

user_embs = user_embedding_model.predict(test_user_model_input, batch_size=2 ** 12)
# user_embs = user_embs[:, i, :]  # i in [0,k_max) if MIND
item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12)

print(user_embs.shape)
print(item_embs.shape)